In [2]:
import os
import shutil
import pandas as pd

# Set iteration of study and source path
study = 'Study5.0Pilot2'
condition = 'Accommodate'
source_dir = "/Users/sm6511/Downloads/acc_5/"

dest_dir = (
    f"/Users/sm6511/Desktop/Prediction-Accomodation-Exp/"
    f"data/{study}/{condition}"
)

target_dates = [
    "2026-05-29"
]

DELETE_FAILED_ATTENTION = False

completion_col = "button_end.numClicks"

if condition.lower() == "predict":
    attention_col = "answer_3_right.numClicks"
elif condition.lower() == "accommodate":
    attention_col = "button_3_correct.numClicks"
else:
    raise ValueError("condition must be 'Predict' or 'Accommodate'")

os.makedirs(dest_dir, exist_ok=True)

moved = []
skipped = []
deleted_failed_attention = []

# collect files sorted by earlier date
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 1: delete failed attention checks
if DELETE_FAILED_ATTENTION:
    for fname in candidate_files:
        src_path = os.path.join(source_dir, fname)

        try:
            df = pd.read_csv(src_path)
            df.columns = df.columns.str.strip()
        except Exception as e:
            print(f"Could not read {fname}: {e}")
            skipped.append((fname, "read error during attention check"))
            continue

        if attention_col not in df.columns:
            print(f"Missing {attention_col} in {fname}")
            skipped.append((fname, "missing attention column"))
            continue

        passed_attention = (
            pd.to_numeric(df[attention_col], errors="coerce")
            .eq(1)
            .any()
        )

        if not passed_attention:
            print(f"🗑️ DELETING FAILED ATTENTION CHECK: {fname}")
            os.remove(src_path)
            deleted_failed_attention.append(fname)

# refresh list of files
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 2: move completed files, earliest first
for fname in candidate_files:
    src_path = os.path.join(source_dir, fname)

    try:
        df = pd.read_csv(src_path)
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"Could not read {fname}: {e}")
        skipped.append((fname, "read error"))
        continue

    if completion_col not in df.columns:
        print(f"Missing {completion_col} in {fname}")
        skipped.append((fname, "missing completion column"))
        continue

    is_complete = (
        pd.to_numeric(df[completion_col], errors="coerce")
        .eq(1)
        .any()
    )

    if is_complete:
        dest_path = os.path.join(dest_dir, fname)
        shutil.move(src_path, dest_path)
        moved.append(fname)
        print(f"MOVED: {fname}")
    else:
        skipped.append((fname, f"{completion_col} != 1"))

print("\n===== SUMMARY =====")

print(f"Deleted failed attention files ({len(deleted_failed_attention)}):")
for f in deleted_failed_attention:
    print(f"  {f}")

print(f"\nMoved files ({len(moved)}):")
for f in moved:
    print(f"  {f}")

print(f"\nSkipped files ({len(skipped)}):")
for f, reason in skipped:
    print(f"  {f} — {reason}")

MOVED: 001_explain2_2026-05-29_12h06.50.915.csv
MOVED: 002_explain2_2026-05-29_12h46.09.528.csv
Missing button_end.numClicks in 003_explain2_2026-05-29_12h46.07.295.csv
Could not read 003_explain2_2026-05-29_12h49.25.390.csv: No columns to parse from file
MOVED: 003_explain2_2026-05-29_12h49.50.663.csv
MOVED: 004_explain2_2026-05-29_12h46.09.803.csv
MOVED: 005_explain2_2026-05-29_09h46.45.495.csv
MOVED: 006_explain2_2026-05-29_09h46.24.321.csv
MOVED: 007_explain2_2026-05-29_12h48.55.944.csv
MOVED: 008_explain2_2026-05-29_12h47.09.736.csv
MOVED: 009_explain2_2026-05-29_12h47.23.556.csv
Missing button_end.numClicks in 010_explain2_2026-05-29_12h54.39.217.csv
MOVED: 010_explain2_2026-05-29_13h11.43.243.csv
MOVED: 011_explain2_2026-05-29_13h00.46.029.csv
MOVED: 012_explain2_2026-05-29_11h48.48.321.csv
MOVED: 012_explain2_2026-05-29_13h07.47.915.csv
MOVED: 013_explain2_2026-05-29_11h48.14.196.csv
MOVED: 014_explain2_2026-05-29_10h53.57.490.csv
MOVED: 015_explain2_2026-05-29_11h09.48.840.csv